In [23]:
import pandas as pd

### 2. Modelo de Pontuação (Regra de Negócio)

Você já trouxe os pesos. Abaixo, organizo em “tabela” para virar fácil regra de cálculo na API/Pandas.

#### 2.1. Scouts de Defesa

- `DS` – Desarmes: **+1,2**
- `FC` – Falta cometida: **-0,3**
- `GC` – Gol contra: **-3,0**
- `CA` – Cartão amarelo: **-1,0**
- `CV` – Cartão vermelho: **-3,0**
- `SG` – Jogo sem sofrer gol  
  - **+5,0** (apenas para **Goleiro, Zagueiro, Lateral**)
- `DE` – Defesa difícil  
  - **+1,0** (apenas **Goleiro**)
- `DP` – Defesa de pênalti  
  - **+7,0** (apenas **Goleiro**)
- `GS` – Gol sofrido  
  - **-1,0** (apenas **Goleiro**)
- `PC` – Pênalti cometido: **-1,0**

#### 2.2. Scouts de Ataque

- `FS` – Falta sofrida: **+0,5**
- `PE` – Passe incompleto: **-0,1**
- `A` – Assistência: **+5,0**
- `FT` – Finalização na trave: **+3,0**
- `FD` – Finalização defendida: **+1,2**
- `FF` – Finalização para fora: **+0,8**
- `G` – Gol: **+8,0**
- `I` – Impedimento: **-0,1**
- `PP` – Pênalti perdido: **-4,0**
- `PS` – Pênalti sofrido: **+1,0**

In [24]:
df_exemplo = pd.read_csv('./cartola/rodada-1.csv')
print("Amostra de dados:")
print(df_exemplo.head(3))
print("\n\nInfo do dataframe:")
print(df_exemplo.info())

Amostra de dados:
   Unnamed: 0  atletas.rodada_id  \
0           0                  1   
1           1                  1   
2           2                  1   

                                        atletas.foto  atletas.status_id  \
0  https://s3.glbimg.com/v1/AUTH_58d78b787ec34892...                  7   
1  https://s3.glbimg.com/v1/AUTH_58d78b787ec34892...                  7   
2  https://s3.glbimg.com/v1/AUTH_58d78b787ec34892...                  7   

   atletas.pontos_num  atletas.variacao_num  atletas.media_num  atletas.slug  \
0                   0                     0                  0  mano-menezes   
1                   0                     0                  0     leo-conde   
2                   0                     0                  0         fabio   

  atletas.apelido  atletas.atleta_id                    atletas.nome  \
0    Mano Menezes              37281  Luis Antônio Venker de Menezes   
1       Léo Condé              37457        Leonardo Rodrigues Condé   

## Implementação do Modelo de Pontuação

**Observação:** Os arquivos CSV contêm apenas a pontuação final (`atletas.pontos_num`), mas não os scouts individuais.  
Abaixo, defino os pesos para referência e análise futura.

In [25]:
SCOUTS_PESOS = {
    'DS': 1.2,   # Desarmes
    'FC': -0.3,  # Falta cometida
    'GC': -3.0,  # Gol contra
    'CA': -1.0,  # Cartão amarelo
    'CV': -3.0,  # Cartão vermelho
    'SG': 5.0,   # Jogo sem sofrer gol (Goleiro, Zagueiro, Lateral)
    'DE': 1.0,   # Defesa difícil (apenas Goleiro)
    'DP': 7.0,   # Defesa de pênalti (apenas Goleiro)
    'GS': -1.0,  # Gol sofrido (apenas Goleiro)
    'PC': -1.0,  # Pênalti cometido
    
    'FS': 0.5,   # Falta sofrida
    'PE': -0.1,  # Passe incompleto
    'A': 5.0,    # Assistência
    'FT': 3.0,   # Finalização na trave
    'FD': 1.2,   # Finalização defendida
    'FF': 0.8,   # Finalização para fora
    'G': 8.0,    # Gol
    'I': -0.1,   # Impedimento
    'PP': -4.0,  # Pênalti perdido
    'PS': 1.0    # Pênalti sofrido
}

# posições que recebem bônus por não sofrer gol (SG)
POSICOES_COM_SG = [1, 2, 3]  # 1=Goleiro, 2=Lateral, 3=Zagueiro

# scouts exclusivos do goleiro
SCOUTS_GOLEIRO = ['DE', 'DP', 'GS']

print("Pesos dos scouts definidos com sucesso!")
print(f"Total de scouts configurados: {len(SCOUTS_PESOS)}")

Pesos dos scouts definidos com sucesso!
Total de scouts configurados: 20


## Análise: Jogador que mais pontuou em cada rodada

In [26]:
# encontrar o jogador que mais pontuou em cada rodada
resultados = []

for rodada in range(1, 39):
    # ler dados da rodada
    df = pd.read_csv(f'./cartola/rodada-{rodada}.csv')
    
    # filtrar jogadores que entraram em campo
    df_jogaram = df[df['atletas.entrou_em_campo'] == True]
    
    if len(df_jogaram) > 0:
        # encontrar o jogador com maior pontuação
        idx_max = df_jogaram['atletas.pontos_num'].idxmax()
        melhor = df_jogaram.loc[idx_max]
        
        resultados.append({
            'Rodada': rodada,
            'Atleta': melhor['atletas.apelido'],
            'Clube': melhor['atletas.clube.id.full.name'],
            'Posição ID': melhor['atletas.posicao_id'],
            'Pontuação': melhor['atletas.pontos_num']
        })

# criar DataFrame com os resultados
df_resultados = pd.DataFrame(resultados)

# mapear IDs de posição para nomes
posicoes = {
    1: 'Goleiro',
    2: 'Lateral',
    3: 'Zagueiro',
    4: 'Meia',
    5: 'Atacante',
    6: 'Técnico'
}
df_resultados['Posição'] = df_resultados['Posição ID'].map(posicoes)

print("Top 10 melhores pontuações individuais por rodada:")
print(df_resultados.nlargest(10, 'Pontuação')[['Rodada', 'Atleta', 'Clube', 'Posição', 'Pontuação']])
print(f"\nTotal de rodadas analisadas: {len(df_resultados)}")

Top 10 melhores pontuações individuais por rodada:
    Rodada           Atleta Clube   Posição  Pontuação
19      21      Samuel Lino   FLA  Atacante       33.1
6        8           Lucero   FOR  Atacante       26.5
35      37           Neymar   SAN      Meia       26.3
11      13       Kaio Jorge   CRU  Atacante       26.2
23      25   Paulo Henrique   VAS   Lateral       26.2
4        6            Pedro   FLA  Atacante       26.0
21      23      Vitor Roque   PAL  Atacante       25.9
26      28      Vitor Roque   PAL  Atacante       25.8
24      26      Flaco López   PAL  Atacante       25.2
32      34  Gabriel Taliari   JUV  Atacante       25.2

Total de rodadas analisadas: 37


In [27]:
# análise das posições que mais pontuam
print("\nDistribuição de posições entre os melhores da rodada:")
print(df_resultados['Posição'].value_counts())

print("\nEstatísticas de pontuação por posição:")
print(df_resultados.groupby('Posição')['Pontuação'].agg(['mean', 'max', 'min', 'count']).round(2))


Distribuição de posições entre os melhores da rodada:
Posição
Atacante    24
Meia         7
Lateral      6
Name: count, dtype: int64

Estatísticas de pontuação por posição:
           mean   max   min  count
Posição                           
Atacante  22.70  33.1  15.0     24
Lateral   21.07  26.2  18.5      6
Meia      20.49  26.3  16.5      7


## Processamento e Salvamento dos Dados com Análises

Vamos adicionar colunas úteis aos dados de cada rodada e salvar os CSVs atualizados.

In [28]:
import os

# criar diretório para salvar os dados processados
output_dir = './cartola_processado'
os.makedirs(output_dir, exist_ok=True)

# mapear posições
posicoes_map = {
    1: 'Goleiro',
    2: 'Lateral',
    3: 'Zagueiro',
    4: 'Meia',
    5: 'Atacante',
    6: 'Técnico'
}

# processar cada rodada
rodadas_processadas = []

for rodada_num in range(1, 39):
    # ler dados da rodada
    df_rodada = pd.read_csv(f'./cartola/rodada-{rodada_num}.csv')
    
    # adicionar nome da posição
    df_rodada['posicao_nome'] = df_rodada['atletas.posicao_id'].map(posicoes_map)
    
    # calcular ranking de pontuação na rodada (apenas para quem jogou)
    df_rodada['ranking_rodada'] = 0
    mask_jogou = df_rodada['atletas.entrou_em_campo'] == True
    df_rodada.loc[mask_jogou, 'ranking_rodada'] = df_rodada.loc[mask_jogou, 'atletas.pontos_num'].rank(
        method='min', ascending=False
    ).astype(int)
    
    # calcular percentil de pontuação (0-100)
    df_rodada['percentil_pontuacao'] = 0.0
    if mask_jogou.sum() > 0:
        df_rodada.loc[mask_jogou, 'percentil_pontuacao'] = (
            df_rodada.loc[mask_jogou, 'atletas.pontos_num'].rank(pct=True) * 100
        ).round(2)
    
    # adicionar classificação de desempenho
    def classificar_desempenho(row):
        if not row['atletas.entrou_em_campo']:
            return 'Não Jogou'
        pontos = row['atletas.pontos_num']
        if pontos >= 15:
            return 'Excelente'
        elif pontos >= 10:
            return 'Muito Bom'
        elif pontos >= 5:
            return 'Bom'
        elif pontos >= 0:
            return 'Regular'
        else:
            return 'Ruim'
    
    df_rodada['classificacao_desempenho'] = df_rodada.apply(classificar_desempenho, axis=1)
    
    # adicionar flag de destaque (top 10% da rodada)
    df_rodada['destaque_rodada'] = False
    if mask_jogou.sum() > 0:
        top_10_percent = df_rodada.loc[mask_jogou, 'atletas.pontos_num'].quantile(0.90)
        df_rodada.loc[mask_jogou, 'destaque_rodada'] = df_rodada.loc[mask_jogou, 'atletas.pontos_num'] >= top_10_percent
    
    # salvar CSV processado
    output_file = f'{output_dir}/rodada-{rodada_num}_processado.csv'
    df_rodada.to_csv(output_file, index=False)
    
    rodadas_processadas.append({
        'Rodada': rodada_num,
        'Total Jogadores': len(df_rodada),
        'Jogaram': mask_jogou.sum(),
        'Maior Pontuação': df_rodada.loc[mask_jogou, 'atletas.pontos_num'].max() if mask_jogou.sum() > 0 else 0,
        'Média Pontuação': df_rodada.loc[mask_jogou, 'atletas.pontos_num'].mean() if mask_jogou.sum() > 0 else 0
    })

# criar resumo do processamento
df_resumo = pd.DataFrame(rodadas_processadas)
df_resumo.to_csv(f'{output_dir}/resumo_rodadas.csv', index=False)

print(f"processamento concluído!")
print(f"arquivos salvos em: {output_dir}/")
print(f"total de rodadas processadas: {len(rodadas_processadas)}")
print(f"\ntop 5 rodadas com maior pontuação:")
print(df_resumo.nlargest(5, 'Maior Pontuação')[['Rodada', 'Jogaram', 'Maior Pontuação', 'Média Pontuação']].round(2))

processamento concluído!
arquivos salvos em: ./cartola_processado/
total de rodadas processadas: 38

top 5 rodadas com maior pontuação:
    Rodada  Jogaram  Maior Pontuação  Média Pontuação
20      21      333             33.1             4.47
7        8      334             26.5             4.02
36      37      268             26.3             3.92
12      13      264             26.2             3.45
24      25      336             26.2             3.54


In [29]:
# exemplo dos dados processados
df_exemplo_processado = pd.read_csv('./cartola_processado/rodada-8_processado.csv')

# filtrar apenas jogadores que entraram em campo e mostrar top 10
df_top10 = df_exemplo_processado[df_exemplo_processado['atletas.entrou_em_campo'] == True].nlargest(10, 'atletas.pontos_num')

print("top 10 jogadores da Rodada 8 com novas colunas:")
print(df_top10[['atletas.apelido', 'posicao_nome', 'atletas.clube.id.full.name', 
                'atletas.pontos_num', 'ranking_rodada', 'percentil_pontuacao', 
                'classificacao_desempenho', 'destaque_rodada']].to_string(index=False))

print(f"\n\nnovas colunas adicionadas:")
print("  • posicao_nome: Nome da posição do jogador")
print("  • ranking_rodada: Posição no ranking da rodada")
print("  • percentil_pontuacao: Percentil de pontuação (0-100)")
print("  • classificacao_desempenho: Excelente/Muito Bom/Bom/Regular/Ruim")
print("  • destaque_rodada: Top 10% da rodada (True/False)")

top 10 jogadores da Rodada 8 com novas colunas:
atletas.apelido posicao_nome atletas.clube.id.full.name  atletas.pontos_num  ranking_rodada  percentil_pontuacao classificacao_desempenho  destaque_rodada
         Lucero     Atacante                        FOR                26.5               1               100.00                Excelente             True
Matheus Pereira         Meia                        CRU                25.6               2                99.70                Excelente             True
     Pochettino         Meia                        FOR                23.7               3                99.40                Excelente             True
     Kaio Jorge     Atacante                        CRU                17.7               4                99.10                Excelente             True
  Renato Kayzer     Atacante                        VIT                16.9               5                98.80                Excelente             True
       Cuiabano      L

## Criação de CSV Consolidado para API

Vamos criar um CSV otimizado para alimentar todos os endpoints da API solicitada.

In [30]:
df_check = pd.read_csv('./cartola/rodada-8.csv')
print("todas as colunas disponíveis:")
for i, col in enumerate(df_check.columns, 1):
    print(f"{i:2d}. {col}")

# 
scouts_cols = [col for col in df_check.columns if any(scout in col.upper() for scout in ['SCOUT', 'DS', 'FC', 'GC', 'CA', 'CV', 'SG', 'DE', 'DP', 'GS', 'PC', 'FS', 'PE', 'FT', 'FD', 'FF'])]
print(f"\ncolunas relacionadas a scouts encontradas: {len(scouts_cols)}")
if scouts_cols:
    print(scouts_cols)

todas as colunas disponíveis:
 1. Unnamed: 0
 2. atletas.status_id
 3. atletas.pontos_num
 4. atletas.atleta_id
 5. atletas.variacao_num
 6. atletas.foto
 7. atletas.apelido
 8. atletas.posicao_id
 9. atletas.jogos_num
10. atletas.clube.id.full.name
11. atletas.nome
12. atletas.preco_num
13. atletas.minimo_para_valorizar
14. atletas.entrou_em_campo
15. atletas.media_num
16. atletas.slug
17. atletas.apelido_abreviado
18. atletas.clube_id
19. atletas.rodada_id
20. DS
21. FC
22. FD
23. FF
24. FS
25. G
26. CA
27. I
28. DE
29. DP
30. GS
31. SG
32. A
33. FT
34. V
35. PS
36. PC
37. CV
38. PP
39. GC

colunas relacionadas a scouts encontradas: 19
['atletas.variacao_num', 'atletas.apelido', 'atletas.posicao_id', 'atletas.entrou_em_campo', 'atletas.apelido_abreviado', 'DS', 'FC', 'FD', 'FF', 'FS', 'CA', 'DE', 'DP', 'GS', 'SG', 'FT', 'PC', 'CV', 'GC']


In [ ]:
# verificar os scouts da rodada 8
print("\nexemplo de dados com scouts (Top 5 jogadores da rodada 8):")
scouts_exemplo = df_check[df_check['atletas.entrou_em_campo'] == True].nlargest(5, 'atletas.pontos_num')
colunas_ver = ['atletas.apelido', 'atletas.clube.id.full.name', 'atletas.pontos_num', 'G', 'A', 'DS', 'FC', 'SG']
print(scouts_exemplo[colunas_ver].to_string(index=False))


exemplo de dados com scouts (Top 5 jogadores da rodada 8):
atletas.apelido atletas.clube.id.full.name  atletas.pontos_num   G   A   DS   FC  SG
         Lucero                        FOR                26.5 3.0 1.0  4.0  8.0 NaN
Matheus Pereira                        CRU                25.6 2.0 1.0 17.0 11.0 NaN
     Pochettino                        FOR                23.7 1.0 1.0  5.0  NaN NaN
     Kaio Jorge                        CRU                17.7 5.0 2.0 12.0  5.0 NaN
  Renato Kayzer                        VIT                16.9 2.0 NaN  NaN  NaN NaN


In [32]:
import glob

# listar todos os arquivos processados
arquivos_processados = sorted(glob.glob('./cartola_processado/rodada-*_processado.csv'))

# concatenar todos os dataframes
lista_dfs = []
for arquivo in arquivos_processados:
    df_temp = pd.read_csv(arquivo)
    # extrair número da rodada do nome do arquivo
    rodada = int(arquivo.split('rodada-')[1].split('_')[0])
    df_temp['rodada'] = rodada
    lista_dfs.append(df_temp)

# criar dataframe consolidado
df_api = pd.concat(lista_dfs, ignore_index=True)

# renomear colunas para facilitar uso na API
df_api = df_api.rename(columns={
    'atletas.atleta_id': 'id_jogador',
    'atletas.apelido': 'apelido',
    'atletas.foto': 'foto',
    'atletas.posicao_id': 'posicao_id',
    'atletas.clube.id.full.name': 'clube_nome',
    'atletas.pontos_num': 'pontuacao',
    'atletas.preco_num': 'preco',
    'atletas.variacao_num': 'variacao_preco',
    'atletas.media_num': 'media_pontos',
    'atletas.jogos_num': 'jogos',
    'atletas.scout.G': 'gols',
    'atletas.scout.A': 'assistencias',
    'atletas.scout.SG': 'sem_sofrer_gol',
    'atletas.entrou_em_campo': 'entrou_em_campo'
})

# adicionar sigla do clube (extrair das 3 primeiras letras do nome)
def obter_sigla_clube(nome_clube):
    mapeamento_clubes = {
        'Atlético-MG': 'CAM', 'Athletico-PR': 'CAP', 'Atlético-GO': 'ACG',
        'Bahia': 'BAH', 'Botafogo': 'BOT', 'Bragantino': 'RBB',
        'Corinthians': 'COR', 'Coritiba': 'CFC', 'Criciúma': 'CRI',
        'Cruzeiro': 'CRU', 'Cuiabá': 'CUI', 'Flamengo': 'FLA',
        'Fluminense': 'FLU', 'Fortaleza': 'FOR', 'Grêmio': 'GRE',
        'Internacional': 'INT', 'Juventude': 'JUV', 'Palmeiras': 'PAL',
        'São Paulo': 'SAO', 'Vasco': 'VAS', 'Vitória': 'VIT'
    }
    return mapeamento_clubes.get(nome_clube, nome_clube[:3].upper())

df_api['clube_sigla'] = df_api['clube_nome'].apply(obter_sigla_clube)

# salvar CSV consolidado
output_api = './cartola_api_completo.csv'
df_api.to_csv(output_api, index=False)

print(f"CSV consolidado criado com sucesso!")
print(f"Total de registros: {len(df_api):,}")

CSV consolidado criado com sucesso!
Total de registros: 28,589


In [33]:
print("CSV da API criado com sucesso!")
print(f"Arquivo: {output_api}")
print(f"Total de registros: {len(df_api):,}")
print(f"Jogadores únicos: {df_api['id_jogador'].nunique():,}")
print(f"Clubes únicos: {df_api['clube_sigla'].nunique()}")
print(f"Rodadas: {df_api['rodada'].min()} a {df_api['rodada'].max()}")
print(f"\nColunas disponíveis: {len(df_api.columns)}")

CSV da API criado com sucesso!
Arquivo: ./cartola_api_completo.csv
Total de registros: 28,589
Jogadores únicos: 980
Clubes únicos: 20
Rodadas: 1 a 38

Colunas disponíveis: 46


In [34]:
# visualizar estrutura do CSV da API
print("Estrutura do CSV da API:")
print(df_api.head(3).to_string())
print(f"\nColunas disponíveis ({len(df_api.columns)}):")
for i, col in enumerate(df_api.columns, 1):
    print(f"{i:2d}. {col}")

Estrutura do CSV da API:
   Unnamed: 0 clube_nome  pontuacao                    atletas.nome  atletas.status_id  jogos  id_jogador atletas.apelido_abreviado       apelido  variacao_preco  posicao_id  atletas.slug  atletas.minimo_para_valorizar  atletas.clube_id  atletas.rodada_id                                                                                                  foto  preco  media_pontos  entrou_em_campo   CA  FC  FF   G   I  DS  FS  FD    DE    GS   A  FT  CV  PS  DP   SG    V  PC  PP  GC posicao_nome  ranking_rodada  percentil_pontuacao classificacao_desempenho  destaque_rodada  rodada clube_sigla
0           0        GRE       6.85  Luis Antônio Venker de Menezes                  7      6       37281                M. Menezes  Mano Menezes            0.80           6  mano-menezes                            NaN               284                 10  https://s3.glbimg.com/v1/AUTH_58d78b787ec34892b5aaa0c7a146155f/clubes_2025/silhuetas/GRE/FORMATO.png   9.74          4.98  